# Transformers: Arquitectura de Atención para Traducción Automática

En esta notebook, exploraremos la implementación de un modelo **Transformer** desde cero utilizando PyTorch. Esta arquitectura revolucionaria, introducida en el paper "Attention is All You Need" (Vaswani et al., 2017), marcó un antes y un después en el campo del Procesamiento de Lenguaje Natural (NLP), siendo la base de modelos modernos como BERT y GPT.

## Introducción

### Objetivos

1. **Comprender la arquitectura Transformer** y cómo funciona el mecanismo de atención (self-attention) para procesar secuencias de texto.
2. **Implementar desde cero los componentes clave** del Transformer: Positional Encoding, Multi-Head Attention, Encoder, y Decoder.
3. **Entrenar un modelo de traducción automática** inglés-español utilizando la arquitectura Transformer completa.
4. **Explorar el uso de máscaras** para el entrenamiento autorregresivo y el manejo de padding en secuencias de longitud variable.

### Contenido

1. Introducción a la arquitectura Transformer y su relevancia en NLP moderno.
2. Preparación de datos y construcción de vocabularios para traducción automática.
3. Implementación del **Positional Encoding** para capturar información de posición en las secuencias.
4. Desarrollo del mecanismo de **Multi-Head Attention** y comprensión del scaled dot-product attention.
5. Construcción de las capas del **Encoder** y **Decoder** con sus componentes: self-attention, encoder-decoder attention, y feed-forward networks.
6. Implementación de **máscaras** para padding y look-ahead masking en el decoder.
7. Entrenamiento del modelo Transformer completo y evaluación en traducción de frases.

### Concepto de Transformer

La arquitectura Transformer revolucionó el procesamiento de secuencias al **eliminar completamente las redes recurrentes** (RNNs y LSTMs) y basarse únicamente en mecanismos de atención. A diferencia de las arquitecturas secuenciales tradicionales, el Transformer puede procesar todos los elementos de una secuencia en paralelo, lo que mejora significativamente la eficiencia del entrenamiento.

**Componentes principales:**

- **Encoder**: Procesa la secuencia de entrada mediante capas de self-attention y redes feed-forward, generando representaciones contextualizadas de cada token.
- **Decoder**: Genera la secuencia de salida de forma autorregresiva, utilizando tanto self-attention sobre los tokens ya generados como atención cruzada (cross-attention) sobre la salida del encoder.
- **Multi-Head Attention**: Permite al modelo atender a diferentes representaciones y posiciones de la secuencia simultáneamente, capturando relaciones complejas entre palabras.
- **Positional Encoding**: Como el Transformer no tiene noción inherente del orden de las palabras, se añade información posicional mediante funciones seno y coseno.

Esta arquitectura se ha convertido en el estándar de facto para tareas de NLP, siendo la base de los modelos de lenguaje más avanzados de la actualidad.

<div align="center">
    <img src="https://d1.awsstatic.com/GENAI-1.151ded5440b4c997bac0642ec669a00acff2cca1.png" width="600px">
</div>

### Dataset de Traducción

Para esta notebook, utilizaremos el dataset de traducción inglés-español de Tatoeba, que contiene pares de frases en ambos idiomas. El objetivo es entrenar un modelo Transformer que aprenda a traducir frases del inglés al español. El dataset incluye frases cortas y medianas de diversos contextos cotidianos, lo que permite al modelo aprender patrones lingüísticos variados.

El dataset está disponible en: [Tatoeba Downloads](https://tatoeba.org/en/downloads)

### Referencias

- [Attention is All You Need](https://arxiv.org/abs/1706.03762) - Vaswani et al. (2017)
- [The Illustrated Transformer](http://jalammar.github.io/illustrated-transformer/) - Jay Alammar <- Encoder + Decoder
- [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) - Visualización interactiva <- Decoder only
- [LLM Visualization](https://bbycroft.net/llm) - Brendan Bycroft <- Decoder only

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

from torchinfo import summary

import math
import numpy as np
import os
import re
from pathlib import Path
from collections import Counter

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import sys

# definimos el dispositivo que vamos a usar
DEVICE = "cpu"  # por defecto, usamos la CPU
if torch.cuda.is_available():
    DEVICE = "cuda"  # si hay GPU, usamos la GPU
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # si no hay GPU, pero hay MPS, usamos MPS
elif torch.xpu.is_available():
    DEVICE = "xpu"  # si no hay GPU, pero hay XPU, usamos XPU

print(f"Usando {DEVICE}")

NUM_WORKERS = 0 # Win y MacOS pueden tener problemas con múltiples workers
if sys.platform == 'linux':
    NUM_WORKERS = 4  # numero de workers para cargar los datos (depende de cada caso)

print(f"Usando {NUM_WORKERS}")

## Cargar los Datos & Preprocesamiento

Vamos a leer el dataset de traducción y realizar un preprocesamiento básico para limpiar y normalizar los textos antes de alimentarlos al modelo. Por ejemplo, vamos a convertir los textos a minúsculas, eliminar algunos caracteres especiales y filtrar las oraciones más largas.

In [ ]:
def clean_text(text):
    # Convertimos a minúsculas
    text = text.lower()
    
    # Insertamos espacios alrededor de los símbolos de puntuación que queremos conservar
    text = re.sub(r'([¿?¡!,])', r' \1 ', text)

    # Eliminamos todo lo que no sea letras, números, o los símbolos que queremos conservar
    text = re.sub(r"[^a-zA-Z0-9áéíóúüñ¿?¡!,]+", ' ', text)

    # Remover espacios extras
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
DATA_PATH = str(Path("data") / "English-Spanish.tsv")

MAX_SENTENCE_LENGTH = 15 # Máxima longitud de las frases que vamos a considerar

def load_data(source_file, max_words=5):
    with open(source_file, "r") as f:
        lines = f.readlines()
    
    # Separamos las frases en dos listas
    input_texts = []
    target_texts = []
    
    for line in lines:
        elements = line.split("\t")

        input_text = elements[1]
        target_text = elements[3]

        input_text_clean = clean_text(input_text)
        target_text_clean = clean_text(target_text)

        # Filtramos frases de hasta max_words palabras
        if len(input_text_clean.split()) <= max_words and len(target_text_clean.split()) <= max_words:
            input_texts.append(input_text_clean)
            target_texts.append(target_text_clean)
    
    return input_texts, target_texts

src_texts, trg_texts = load_data(DATA_PATH, MAX_SENTENCE_LENGTH)

In [ ]:
print(f"Number of samples: {len(src_texts)}")

In [ ]:
random_idx = np.random.randint(0, len(src_texts), 10)
for idx in random_idx:
    print(f"Input: {src_texts[idx]}")
    print(f"Target: {trg_texts[idx]}\n")

## Construcción de los Vocabularios

Es importante construir un vocabulario para cada idioma en el dataset, ya que cada vocabulario tiene que ser capaz de mapear palabras a índices enteros y viceversa.

> Nota: para reducir el tiempo de entrenamiento, vamos a limitar el tamaño del vocabulario a las palabras más comunes en cada idioma, con el argumento `FREQ_THRESHOLD` controlamos la cantidad de palabras que se incluirán en el vocabulario.

Tenemos además que agregar token especiales:

- `SOS` (Start of Sentence): Indica el inicio de una oración.
- `EOS` (End of Sentence): Indica el final de una oración.
- `UNK` (Unknown): Indica una palabra desconocida que no está en el vocabulario.
- `PAD` (Padding): Se utiliza para rellenar secuencias a la misma longitud.

In [ ]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
FREQ_THRESHOLD = 1 # Frecuencia mínima para considerar una palabra en el vocabulario

class Vocab:
    def __init__(self):
        # mapea palabras a índices
        self.word2index = {}
        # mapea índices a palabras
        self.index2word = {}
        # contador de palabras
        self.word_count = Counter()
        self.index = 0

        # Tokens especiales
        self.add_special_tokens()

    def add_special_tokens(self):
        self.add_word(PAD_TOKEN)
        self.add_word(SOS_TOKEN)
        self.add_word(EOS_TOKEN)
        self.add_word(UNK_TOKEN)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.index
            self.index2word[self.index] = word
            self.index += 1

    def build_vocab(self, sentences, min_freq=1):
        word_counter = Counter()
        for sentence in sentences:
            for word in sentence.split():
                word_counter[word] += 1

        # Filtrar palabras que no alcanzan la frecuencia mínima
        words = [word for word, count in word_counter.items() if count >= min_freq]

        # Agregar palabras filtradas al vocabulario
        for word in words:
            self.add_word(word)
            self.word_count[word] = word_counter[word]

    def __len__(self):
        return len(self.word2index)
        
    def __getitem__(self, key):
        if isinstance(key, int):
            return self.index2word.get(key, UNK_TOKEN)
        if isinstance(key, str):
            return self.word2index.get(key, self.word2index[UNK_TOKEN])


# Construimos los vocabularios
SRC_VOCAB = Vocab()
TRG_VOCAB = Vocab()

SRC_VOCAB.build_vocab(src_texts, min_freq=FREQ_THRESHOLD)
TRG_VOCAB.build_vocab(trg_texts, min_freq=FREQ_THRESHOLD)

SRC_VOCAB_SIZE = len(SRC_VOCAB)
TRG_VOCAB_SIZE = len(TRG_VOCAB)

print(f"English vocab size: {SRC_VOCAB_SIZE}")
print(f"Spanish vocab size: {TRG_VOCAB_SIZE}")

Algunos ejemplos de uso de los vocabularios:

- `SRC_VOCAB['hello']`: Devuelve el índice de la palabra "hello" en el vocabulario de origen.
- `TGT_VOCAB['hola']`: Devuelve el índice de la palabra "hola" en el vocabulario de destino.
- `TRG_VOCAB['palabra_no_existente']`: Devuelve el índice de la palabra desconocida (`<UNK>`) en el vocabulario de destino.
- `TRG_VOCAB[10]`: Devuelve la palabra en el índice 10 del vocabulario de destino.
- `TRG_VOCAB[3]`: Devuelve la palabra en el índice 3 del vocabulario de destino.
- `SRC_VOCAB[PAD_TOKEN]`: Devuelve el índice del token de padding en el vocabulario de origen.

In [ ]:
print(SRC_VOCAB["hello"])
print(TRG_VOCAB["hola"])
print(TRG_VOCAB["palabra_no_existente"])
print(TRG_VOCAB[10])
print(TRG_VOCAB[3])
print(TRG_VOCAB[PAD_TOKEN])

In [ ]:
# Función para codificar una frase
def encode_sentence(sentence, vocab):
    return [vocab[word] for word in sentence.split()]

print(encode_sentence("hello world", SRC_VOCAB))
print(encode_sentence("hola mundo extraterrestre", TRG_VOCAB))

In [ ]:
def decode_sentence(indices, vocab):
    return " ".join([vocab[idx] for idx in indices if idx != vocab[PAD_TOKEN] and idx != vocab[EOS_TOKEN] and idx != vocab[SOS_TOKEN]])

print(decode_sentence([10, 11, 12], TRG_VOCAB))

## Dataset de Traducción

Trabajaremos con un pequeño dataset de traducción **inglés-español**, compuesto por pares de frases simples. El objetivo es que el modelo aprenda a traducir una frase en inglés a su equivalente en español.

#### Ejemplos de Pares:

- **Inglés**: "hello" → **Español**: "hola"
- **Inglés**: "how are you?" → **Español**: "¿cómo estás?"

### Preparación del Dataset

Cada frase será tokenizada y convertida a índices numéricos de sus respectivos vocabularios. En las secuencias objetivo, añadimos los tokens especiales `<SOS>` y `<EOS>` para marcar el inicio y el fin de cada traducción.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, source_sentences, target_sentences):
        super(TranslationDataset, self).__init__()
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences
    
    def __len__(self):
        return len(self.source_sentences)
    
    def __getitem__(self, idx):
        source_sentence = self.source_sentences[idx]
        target_sentence = self.target_sentences[idx]
        encoded_source_sentence = encode_sentence(source_sentence, SRC_VOCAB)
        encoded_target_sentence = encode_sentence(target_sentence, TRG_VOCAB)

        # Añadimos tokens especiales <SOS> y <EOS>
        encoded_target_sentence = [TRG_VOCAB[SOS_TOKEN]] + encoded_target_sentence + [TRG_VOCAB[EOS_TOKEN]]
        
        x = torch.tensor(encoded_source_sentence, dtype=torch.long)
        y = torch.tensor(encoded_target_sentence, dtype=torch.long)
        
        return x, y

# Crear el dataset y el dataloader
train_dataset = TranslationDataset(src_texts, trg_texts)
val_len = int(0.10 * len(train_dataset))
train_len = len(train_dataset) - val_len
train_dataset, val_dataset = random_split(train_dataset, [train_len, val_len])

Debido a que las frases tienen longitudes variables, utilizaremos padding para asegurarnos de que todas las secuencias en un batch tengan la misma longitud.

La función `collate_fn` es un argumento opcional que se pasa al DataLoader de PyTorch para personalizar el procesamiento de los datos. En este caso, se utiliza para rellenar y agrupar las secuencias de entrada y salida en lotes.

In [ ]:
def collate_fn(batch):
    sources = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    
    # Padding de las secuencias
    sources_padded = pad_sequence(sources, padding_value=SRC_VOCAB[PAD_TOKEN], batch_first=True, padding_side='right')
    targets_padded = pad_sequence(targets, padding_value=TRG_VOCAB[PAD_TOKEN], batch_first=True, padding_side='right')
    
    return sources_padded, targets_padded

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## Modelo

> **Encoder**: The encoder is composed of a stack of $N = 6$ identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection around each of the two sub-layers, followed by layer normalization. That is, the output of each sub-layer is $LayerNorm(x + Sublayer(x))$, where $Sublayer(x)$ is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension $d_{model} = 512$.
>
> **Decoder**: The decoder is also composed of a stack of $N = 6$ identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position,ensures that the predictions for position $i$ can depend only on the known outputs at positions less than $i$.

Paper original: [Attention is All You Need](https://arxiv.org/abs/1706.03762)

<div align="center">
    <img src="https://d1.awsstatic.com/GENAI-1.151ded5440b4c997bac0642ec669a00acff2cca1.png" width="400px">
</div>

### Positional Encoding

<div style="text-align: center;">
    <img src="https://kazemnejad.com/img/transformer_architecture_positional_encoding/model_arc.jpg" width="1200"/>
</div>

> Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel as the embeddings, so that the two can be summed. There are many choices of positional encodings, learned and fixed. In this work, we use sine and cosine functions of different frequencies:
> $$
> \begin{aligned}
> \text{PE}_{(pos,2i)} &= \sin(pos / 10000^{2i/d_{\text{model}}}) \\
> \text{PE}_{(pos,2i+1)} &= \cos(pos / 10000^{2i/d_{\text{model}}})
> \end{aligned}
> $$
> where $pos$ is the position and $i$ is the dimension. That is, each dimension of the positional encoding corresponds to a sinusoid. The wavelengths form a geometric progression from $2\pi$ to $10000 \cdot 2\pi$. We chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions, since for any fixed offset $k$, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$.

> Nota: $d_{model}$ = embedding dimension

Esta **forma de codificación posicional** permite:
- Dada una posición, obtener una representación única para esa posición (depende de la frecuencia y dimensión).
- La distancia entre dos posiciones es consistente sin importar la longitud de la secuencia.
- El modelo puede generalizar a secuencias más largas sin necesidad de reentrenamiento.
- Su calculo es determinista.

**¿Por qué usar funciones seno y coseno?** Permite al model capturar relaciones de distancia entre posiciones mediante combinaciones lineales. Ver [Linear Relationships in the Transformer’s Positional Encoding](https://blog.timodenk.com/linear-relationships-in-the-transformers-positional-encoding/)

**¿Por qué 10000?** Es un hiperparámetro que define la escala de las frecuencias utilizadas en las funciones seno y coseno. Si fuese muy pequeño, dos posiciones diferentes podrían tener representaciones muy similares, si fuese muy grande, las diferencias entre posiciones cercanas podrían volverse insignificantes. "The wavelengths form a geometric progression from $2\pi$ to $10000 \cdot 2\pi$." -> la primera dimensión (i=0) tendrá una frecuencia alta (ciclos rápidos), cada aproximadamente 6.57 ($2\pi$), mientras que las últimas dimensiones tendrán frecuencias mucho más bajas (ciclos lentos), permitiendo capturar patrones a largo plazo.

**¿Por qué se suman y no concatenan?** 
- La suma mantiene la dimensionalidad constante (baja la complejidad computacional).
- Empiricamente, sumar embeddings y codificaciones posicionales ha demostrado ser efectivo. Ver [Rethinking Positional Encoding in Language Pre-training](https://arxiv.org/abs/2006.15595)

Links útiles:
- [Transformer Architecture: The Positional Encoding](https://kazemnejad.com/blog/transformer_architecture_positional_encoding/)
- [\[video\] ¿Por qué estas REDES NEURONALES son tan POTENTES? ](https://www.youtube.com/watch?v=xi94v_jl26U)
- [\[video\] How do Transformer Models keep track of the order of words? Positional Encoding](https://www.youtube.com/watch?v=IHu3QehUmrQ)


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super(PositionalEncoding, self).__init__()
        assert d_model % 2 == 0, "d_model debe ser par para usar sinusoides"
        # self.register_buffer('pe', pe) # Registrar 'pe' como buffer para que no se actualice durante el entrenamiento : https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
        pass

    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        # pe: [1, max_len, d_model]
        pass

summary(PositionalEncoding(256, MAX_SENTENCE_LENGTH + 2))

No tiene párametros entrenables !

```python
[[ 0.          1.          0.          1.        ]
 [ 0.84147096  0.54030234  0.00999983  0.99995   ]
 [ 0.9092974  -0.41614684  0.01999867  0.9998    ]
 [ 0.14112    -0.9899925   0.0299955   0.99955004]
 [-0.7568025  -0.6536436   0.03998933  0.9992001 ]
 [-0.9589243   0.2836622   0.04997917  0.99875027]]
```

In [ ]:
# Parámetros
d_model = 4
max_len = 6

# Crear instancia de PositionalEncoding
pos_encoding = PositionalEncoding(d_model, max_len)
# Imprimir los valores obtenidos
positional_encoded_values = pos_encoding.pe.squeeze().numpy()
# Mostrar los valores numéricos obtenidos
print(positional_encoded_values)

In [ ]:
rand_embed = torch.rand(1, 5, d_model) # generamos un batch con una sola secuencia de 5 tokens
print(rand_embed, '\n') # embedding original
print(pos_encoding(rand_embed)) # embedding + pe

### Multi-Head Attention

> We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of dimension $d_k$, and values of dimension $d_v$. We compute the dot products of the query with all keys, divide each by $\sqrt{d_k}$, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix $Q$. The keys and values are also packed together into matrices $K$ and $V$ . We compute the matrix of outputs as:
> 
> $$
> \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
> $$
>
> ...
>
> We suspect that for large values of $d_k$, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients. To counteract this effect, we scale the dot products by $\frac{1}{\sqrt{d_k}}$.
>
> ...
> 
> Instead of performing a single attention function with $d_{\text{model}}$-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values $h$ times with different, learned linear projections to $d_k$, $d_k$ and $d_v$ dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding $d_v$-dimensional output values. These are concatenated and once again projected, resulting in the final values:
> 
> $$
> \begin{aligned}
> \text{MultiHead}(Q, K, V) &= \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O \\
> \text{where head}_i &= \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
> \end{aligned}


Links útiles:
- [\[AI by hand\] 11. Self Attention](https://aibyhand.substack.com/p/11-can-you-calculate-self-attention)
- [\[video\] ¿Qué es un TRANSFORMER?](https://www.youtube.com/watch?v=aL-EmKuB078)

<div style="text-align: center;">
    <img src="https://yjucho1.github.io/assets/img/2018-10-13/transformer.png" width="1200"/>
</div>

> The Transformer uses multi-head attention in three different ways:
> - In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models.
> - The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
> - Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super(MultiHeadAttention, self).__init__()
        
        pass

    def forward(self, values, keys, queries, mask):
        # values = [batch_size, values_len, d_model]
        # keys = [batch_size, keys_len, d_model]
        # queries = [batch_size, queries_len, d_model]
        # values_len = keys_len
        pass
    
summary(MultiHeadAttention(d_model=256, heads=8))

### Feed-Forward

> In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully connected feed-forward network, which is applied to each position separately and identically. This consists of two linear transformations with a ReLU activation in between.
> 
> $$
> \text{FFN}(x) = \max(0, xW_1 + b_1) W_2 + b_2
> $$
>
> (...) The dimensionality of input and output is $d_{\text{model}} = 512$, and the inner-layer has dimensionality $d_{\text{ff}} = 2048$.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, expansion_factor):
        super(FeedForward, self).__init__()
        pass

    def forward(self, x):
        pass

### Encoder

<div align="center">
    <img src="https://www.researchgate.net/publication/334288604/figure/fig1/AS:778232232148992@1562556431066/The-Transformer-encoder-structure.ppm" width="400px">
</div>

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout, forward_expansion):
        super(EncoderLayer, self).__init__()
        pass
        
    def forward(self, x, mask):
        # x: [Batch, seq_len, d_model]
        pass

In [ ]:
class Encoder(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        d_model,
        num_layers,
        heads,
        forward_expansion,
        dropout,
        max_length
    ):
        super(Encoder, self).__init__()
        pass
        
    def forward(self, x, mask):
        pass

### Decoder

<div align="center">
    <img src="https://d1.awsstatic.com/GENAI-1.151ded5440b4c997bac0642ec669a00acff2cca1.png" width="400px">
</div>

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, forward_expansion, dropout):
        super(DecoderLayer, self).__init__()
        pass

    def forward(self, x, enc_out, src_mask, trg_mask):
        pass

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        trg_vocab_size,
        d_model,
        num_layers,
        heads,
        forward_expansion,
        dropout,
        max_length
    ):
        super(Decoder, self).__init__()
        pass

    def forward(self, x, enc_out, src_mask, trg_mask):
        pass

### Masking

Para la diagonal superior de la matriz de atención, se aplica una máscara de ceros para que el modelo no pueda ver las palabras futuras.
Nos ayudamos con la función [torch.tril](https://pytorch.org/docs/stable/generated/torch.tril.html) para obtener la matriz triangular inferior de una matriz cuadrada.

```python
[[1, 0, 0, 0],
 [1, 1, 0, 0],
 [1, 1, 1, 0],
 [1, 1, 1, 1]]
```

In [ ]:
torch.tril(torch.full((5, 5), True))

In [ ]:
def create_src_mask(src, src_pad_idx):
        # src: [batch_size, src_len]
        src_mask = (src != src_pad_idx).unsqueeze(1).unsqueeze(2)
        # [batch_size, 1, 1, src_len]
        return src_mask

In [ ]:
input_test = torch.tensor([[1, 2, 3, 4, 0, 0, 0]], dtype=torch.float32)
input_test_mask = create_src_mask(input_test, 0)
print(input_test_mask)

In [ ]:
def create_trg_mask(trg, trg_pad_idx):
    # trg: [batch_size, trg_len]
    trg_pad_mask = (trg != trg_pad_idx).unsqueeze(1).unsqueeze(2)
    # [batch_size, 1, 1, trg_len]
    
    # Crear máscara de look-ahead
    trg_len = trg.size(1)
    trg_sub_mask = torch.tril(torch.full((trg_len, trg_len), True, device=trg.device))
    # [batch_size, 1, trg_len, trg_len]
    trg_mask = trg_pad_mask & trg_sub_mask
    return trg_mask

In [ ]:
input_test = torch.tensor([[1, 2, 3, 4, 0, 0, 0]], dtype=torch.float32)
print(create_trg_mask(input_test, 0))

### Transformer

Links útiles:
- [\[visualización\] LLM Visualization](https://bbycroft.net/llm)
- [\[visualización\] TRANSFORMER EXPLAINER](https://poloclub.github.io/transformer-explainer/)


<div align="center">
    <img src="https://d1.awsstatic.com/GENAI-1.151ded5440b4c997bac0642ec669a00acff2cca1.png" width="400px">
</div>

In [ ]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        d_model=512,
        num_layers=6,
        forward_expansion=4,
        heads=8,
        dropout=0.1,
        max_length=100,
    ):
        super(Transformer, self).__init__()
        pass

    def forward(self, src, trg):
        pass

## Training

In [ ]:
# Parámetros del modelo
SRC_PAD_IDX = SRC_VOCAB[PAD_TOKEN]
TRG_PAD_IDX = TRG_VOCAB[PAD_TOKEN]
D_MODEL = 256
NUM_LAYERS = 4
FORWARD_EXPANSION = 4
HEADS = 4
DROPOUT = 0.1

LR = 0.0005

model = Transformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    trg_vocab_size=TRG_VOCAB_SIZE,
    src_pad_idx=SRC_PAD_IDX,
    trg_pad_idx=TRG_PAD_IDX,
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    forward_expansion=FORWARD_EXPANSION,
    heads=HEADS,
    dropout=DROPOUT,
    max_length=MAX_SENTENCE_LENGTH + 2
).to(DEVICE)


criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
def process_batch(src, trg, model, criterion):
    # Entrada al decoder: todos los tokens excepto el último
    input_trg = trg[:, :-1]
    # input_trg: [batch_size, trg_len - 1]

    # Objetivo para la pérdida: todos los tokens excepto el primero
    target_trg = trg[:, 1:]
    # target_trg: [batch_size, trg_len - 1]

    # Pasar por el modelo
    output = model(src, input_trg)
    # output: [batch_size, trg_len - 1, trg_vocab_size]
    
    # Reestructurar las dimensiones para calcular la pérdida
    output = output.reshape(-1, output.size(-1))
    # output: [batch_size * (trg_len - 1), trg_vocab_size]
    
    target_trg = target_trg.reshape(-1)
    # target_trg: [batch_size * (trg_len - 1)]
    
    return criterion(output, target_trg)

In [ ]:
def evaluate_epoch(model, criterion, valid_loader):
    model.eval()
    valid_loss = 0

    with torch.no_grad():
        for src, trg in valid_loader:
            src = src.to(DEVICE)
            trg = trg.to(DEVICE)
            
            loss = process_batch(src, trg, model, criterion)
            
            valid_loss += loss.item()

    return valid_loss / len(valid_loader)

In [ ]:
def train_epoch(model, criterion, train_loader, optimizer):
    model.train()
    train_loss = 0

    for src, trg in train_loader:
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)
        
        optimizer.zero_grad()
        
        loss = process_batch(src, trg, model, criterion)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    return train_loss / len(train_loader)

In [ ]:
def train(model, train_loader, valid_loader, optimizer, criterion, n_epochs):
    for epoch in range(n_epochs):
        train_loss = train_epoch(model, criterion, train_loader, optimizer)
        val_loss = evaluate_epoch(model, criterion, valid_loader)
        print(f"Epoch {epoch + 1} Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")

In [ ]:
NUM_EPOCHS = 5

train(model, train_loader, valid_loader, optimizer, criterion, NUM_EPOCHS)

In [ ]:
def translate_sentence(model, sentence, max_len):
    # Configuramos el modo de evaluación
    model.eval()

    # Limpiamos la oración de entrada
    sentence = clean_text(sentence)  

    # Preprocesamos la oración de entrada (src_sentence) a tensor
    sentence_indexes = encode_sentence(sentence, SRC_VOCAB)
    # Convertimos a tensor y añadimos una dimensión extra para el batch
    sentence_tensor = torch.tensor(sentence_indexes).unsqueeze(0).to(DEVICE)

    # Creamos una máscara para la entrada (evitamos atender al padding)
    src_mask = create_src_mask(sentence_tensor, SRC_VOCAB[PAD_TOKEN])

    with torch.no_grad():
        # Pasamos por el encoder
        enc_src = model.encoder(sentence_tensor, src_mask)

        # El primer token del decoder es <SOS>
        input_tokens = torch.tensor([TRG_VOCAB[SOS_TOKEN]]).unsqueeze(0).to(DEVICE)

        # Almacenamos los tokens generados por el decoder
        generated_tokens = []

        # Decodificación paso a paso
        for _ in range(max_len):
            trg_mask = create_trg_mask(input_tokens, TRG_VOCAB[PAD_TOKEN])
            
            # Pasamos el token actual por el decoder
            output = model.decoder(input_tokens, enc_src, src_mask, trg_mask)
            
            # Solo nos interesa el último token generado
            output = output[:, -1, :]
            
            # Obtener el token con mayor probabilidad
            top1 = output.argmax(1).item()

            # Si el token predicho es <EOS>, detenemos la decodificación
            if top1 == TRG_VOCAB[EOS_TOKEN]:
                break

            generated_tokens.append(top1)

            # El próximo token de entrada es el que acaba de predecir el modelo
            input_tokens = torch.cat([input_tokens, torch.tensor([[top1]]).to(DEVICE)], dim=1)

        # Convertimos los índices predichos en palabras usando el vocabulario
        predicted_sentence = decode_sentence(generated_tokens, TRG_VOCAB)

    return predicted_sentence

In [ ]:
sentences = [
    "I am hungry",
    "I am tired",
    "I am happy",
    "I'm sad",
    "I am angry",
    "every time I study, I get sleepy",
    "I am going to the gym",
    "I am going to the beach",
    "I am going to the supermarket",
    "I'm going to the movies",
    "I don't know what to do",
    "I love deep learning",
    "I can't open the door",
    "you can go if you want to",
    "i'm going to the party",
    "where does all this come from ?",
    "I can read your mind",
    "I can't believe it",
    "I can't believe you",
    "I can't believe this",
    "I didn't like it",
    "You can do it",
    "Do you speak Italian?",
    "Do you want to learn Spanish?",
    "I want to learn French",
    "Do you want to go to the movies?",
    "this is my favorite song",
    "I can't wait to see you",
    "see you later",
    "have a nice day",
    "we'll talk later",
    "let's grab a coffee sometime",
    "they're coming to the party",
    "she's my best friend",
    "he's a great guy",
    "My class is in 30 minutes.",
    "I have class tomorrow.",
    "This is a very long sentence, let's see how the model handles it.",
    "Can you help me with my homework?",
    "What time is it?",
    "Where is the nearest restaurant?",
    "How do I get to the airport?",
    "I would like to make a reservation."
]

for sentence in sentences:
    print(f"Input: {sentence}")
    print(f"Translation: {translate_sentence(model, sentence, MAX_SENTENCE_LENGTH + 2)}\n")

In [ ]:
translate_sentence(model, "Attention is all you need", MAX_SENTENCE_LENGTH + 2)

## Tarea
Modificar la inferencia para que no sea determinista, es decir, que en lugar de tomar siempre el token con mayor probabilidad, tome muestras de la distribución de probabilidad generada por el modelo en cada paso. Incluir parámetros como temperature y top-k sampling para controlar la diversidad de las traducciones generadas. Ver [torch.multinomial](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html), [torch.topk](https://docs.pytorch.org/docs/stable/generated/torch.topk.html#torch.topk).